# 07b · The two baselines on the final model's training set, with early stopping, and the comparison

Notebook 07 fitted ItemKNN and logistic regression on the bookings before a time cut and trained the logistic regression for a fixed 3 SAGA passes. Here both baselines are refitted on **exactly the training set of the final model (01d)** and compared with it:

- **Training set**: the 1,710,952 warm next-booking targets of the train split plus the 713,754 first bookings (L = 0) inside the train window, i.e. every booking up to the last train target (event index 2,424,705). Counts and event indices are asserted against the 01d result file; eval rows are asserted identical (warm test 213,869; cold test 49,462).
- **ItemKNN** (`implicit` cosine; history only: last 20 bookings before the target). No epochs; the neighbourhood size `K` is chosen on valid. Cold rows have no history, so they fall back to the **first-booking popularity of the training set** (the first bookings it was given), the same information the plain SMLP4Rec learns from its empty-history rows.
- **Logistic regression** (multinomial, SAGA; destination + trip context, no history; one-hot). Trained **one SAGA pass at a time with warm start, up to `max_epochs` passes, with the same early-stopping rule as 01d** (monitor valid warm NDCG@10, `early_stop_patience`, `early_stop_min_delta` from `configs/smlprec_expedia.yaml`); `C` is chosen on valid among 0.1 / 1 / 10 at each `C`'s best epoch. Warm start continues from the previous weights but SAGA's gradient memory restarts every pass, so this approximates one long SAGA run.
- **Selection**: valid warm NDCG@10 for `K`, `C` and the epoch (the metric of 01d); test is scored once with the choice.
- **Comparison** (section 7): both baselines, the plain SMLP4Rec of 01e (no query token) and the final model of 01d (query token, and query token + sameDest), on warm / cold / all-events test rows, read from the result JSONs.

Limitation: `hotel_cluster` is a proxy for room category / package; one dataset; offline only; single seed. Kernel: `nbp`. Output: `results/week4_rebuild/baselines_es_run.json`, `results/week4_rebuild/baselines_vs_final_es.json`; per-row ranks in `data/interim/` (gitignored).

In [1]:
import json
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

import implicit
import sklearn
from implicit.nearest_neighbours import CosineRecommender
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from nbp.models.early_stop import EarlyStopper, mean_nll

BOOKINGS = ROOT / "data" / "interim" / "expedia_bookings.parquet"
RES = ROOT / "results" / "week4_rebuild"
CFG = yaml.safe_load((ROOT / "configs" / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
FINAL_JSON = RES / "smlprec_query_cold_train_es_run.json"  # 01d with early stopping (final model)
FINAL_COLD_JSON = RES / "smlprec_query_cold_L0_cold_train_es.json"
PLAIN_ES_JSON = RES / "smlprec_plain_cold_train_es_run.json"  # 01e
FAST = os.environ.get("NBP_FAST") == "1"  # smoke test: one C, one pass; every output gets the suffix _fast
SUF = "_fast" if FAST else ""
OUT_JSON = RES / f"baselines_es_run{SUF}.json"
CMP_JSON = RES / f"baselines_vs_final_es{SUF}.json"
RANKS_DIR = ROOT / "data" / "interim"

N_CLUSTERS = 100
MAX_HIST = CFG["MAX_ITEM_LIST_LENGTH"]
SPLIT = tuple(CFG["eval_args"]["split"]["RS"])
SEED = int(CFG["seed"])
MAX_EPOCHS, PATIENCE, MIN_DELTA = CFG["max_epochs"], CFG["early_stop_patience"], CFG["early_stop_min_delta"]
KS = (5, 10, 20)
K_GRID = [5, 10, 20, 50, 100]  # ItemKNN neighbours kept per item (100 = all)
C_GRID = [0.1, 1.0, 10.0]  # inverse L2 strength of the logistic regression
if FAST:
    C_GRID, MAX_EPOCHS = [10.0], 1
LEAD_BINS = [-1, 0, 2, 6, 13, 29, 59, 89, 179, 364, 10_000]  # days between search and check-in
STAY_CLIP = 14  # nights; longer stays share one bucket
LR_FEATURES = [  # query-time fields only
    "ctx_dest_id", "ctx_dest_type", "ctx_checkin_month", "stay_bucket", "lead_bucket",
    "ctx_adults", "ctx_children", "ctx_rooms", "ctx_is_package", "ctx_channel", "ctx_is_mobile",
    "ctx_site_name", "ctx_posa_continent", "ctx_user_country",
]
FORBIDDEN = ("tgt_", "ctx_user_city", "ctx_user_region", "distance", "user_id", "item_id", "src_row", "aux_cnt")
print("implicit", implicit.__version__, "| scikit-learn", sklearn.__version__, "|", dict(MAX_EPOCHS=MAX_EPOCHS, PATIENCE=PATIENCE, MIN_DELTA=MIN_DELTA))

implicit 0.7.3 | scikit-learn 1.9.1 | {'MAX_EPOCHS': 10, 'PATIENCE': 2, 'MIN_DELTA': 0.0007}


## 1. Data, split and the final model's training set

In [2]:
def load_bookings() -> pd.DataFrame:
    # bookings sorted by (time, user, src_row); event = row index (= the RecBole event index of 01d);
    # pos = per-user index (0 = first booking); split = global temporal 80/10/10 over next-booking targets (pos > 0)
    d = pd.read_parquet(BOOKINGS, columns=["user_id", "item_id", "timestamp", "src_row", "ctx_dest_id"])
    d["ts"] = ((d["timestamp"] - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)).astype("int64")
    d = d.sort_values(["ts", "user_id", "src_row"], kind="stable").reset_index(drop=True)
    d["item_id"] = d["item_id"].astype("int64")
    d["event"] = np.arange(len(d))
    d["pos"] = d.groupby("user_id").cumcount()
    d["split"] = "first_booking"
    tgt = np.flatnonzero(d["pos"].to_numpy() > 0)
    n_tr, n_va = int(SPLIT[0] * len(tgt)), int(SPLIT[1] * len(tgt))
    d.loc[tgt[:n_tr], "split"] = "train"
    d.loc[tgt[n_tr : n_tr + n_va], "split"] = "valid"
    d.loc[tgt[n_tr + n_va :], "split"] = "test"
    return d


t0 = time.time()
b = load_bookings()
ctx = pd.read_parquet(
    BOOKINGS,
    columns=["src_row", "ctx_dest_type", "ctx_checkin_month", "ctx_stay_nights", "ctx_lead_days",
             "ctx_adults", "ctx_children", "ctx_rooms", "ctx_is_package", "ctx_channel", "ctx_is_mobile",
             "ctx_site_name", "ctx_posa_continent", "ctx_user_country"],
)
assert ctx["src_row"].is_unique
b = b.merge(ctx, on="src_row", how="left", validate="one_to_one")
b = b.sort_values("event", kind="stable").reset_index(drop=True)

final_ref = json.loads(FINAL_JSON.read_text(encoding="utf-8"))
cold_ref = json.loads(FINAL_COLD_JSON.read_text(encoding="utf-8"))["n_rows"]
last = {s: int(b.loc[b["split"] == s, "event"].max()) for s in ("train", "valid")}
assert last["train"] == final_ref["splits"]["train"]["last_target_event"], last
assert last["valid"] == final_ref["splits"]["valid"]["last_target_event"], last

first = b["pos"] == 0
b.loc[first & (b["event"] < last["train"]), "split"] = "train_first"
b.loc[first & (b["event"] >= last["train"]) & (b["event"] < last["valid"]), "split"] = "valid_cold"
b.loc[first & (b["event"] >= last["valid"]), "split"] = "test_cold"
seen_users = set(b.loc[b["split"].isin(["train", "train_first"]), "user_id"].unique())
b["seen_user"] = b["user_id"].isin(seen_users)

counts = b["split"].value_counts().to_dict()
print(counts)
for s in ("train", "valid", "test"):
    assert counts[s] == final_ref["splits"][s]["targets"], (s, counts[s])
assert counts["train_first"] == final_ref["cold_training_rows"]
assert counts["valid_cold"] == cold_ref["valid"] and counts["test_cold"] == cold_ref["test"]
train_rows = b[b["split"].isin(["train", "train_first"])]
assert len(train_rows) == last["train"] + 1 == final_ref["splits"]["train"]["targets"] + final_ref["cold_training_rows"]
print(f"training set: {len(train_rows):,} bookings = {counts['train']:,} warm targets + {counts['train_first']:,} first bookings (identical to 01d); {time.time() - t0:.0f}s")

EVAL = {s: {"warm": b[b["split"] == s].reset_index(drop=True), "cold": b[b["split"] == s + "_cold"].reset_index(drop=True)}
        for s in ("valid", "test")}
Y = {s: {k: v["item_id"].to_numpy() for k, v in d.items()} for s, d in EVAL.items()}

{'train': 1710952, 'train_first': 713754, 'valid': 213869, 'test': 213869, 'valid_cold': 50769, 'test_cold': 49462}


training set: 2,424,706 bookings = 1,710,952 warm targets + 713,754 first bookings (identical to 01d); 6s


## 2. Shared functions: popularity tie-break, ranks, history matrices

In [3]:
pop = np.bincount(train_rows["item_id"].to_numpy(), minlength=N_CLUSTERS).astype(np.float64)
pop_share = pop / pop.sum()
first_train = train_rows[train_rows["split"] == "train_first"]
pop_first = np.bincount(first_train["item_id"].to_numpy(), minlength=N_CLUSTERS).astype(np.float64)
pop_first_share = pop_first / pop_first.sum()  # first-booking popularity: cold fallback of ItemKNN and a reference row
order = np.argsort(-pop, kind="stable")  # column j of a permuted matrix = cluster order[j]; ties go to the more popular cluster
inv_order = np.empty(N_CLUSTERS, np.int64)
inv_order[order] = np.arange(N_CLUSTERS)


def rank_pop_ties(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    # rank of the true cluster; ties go to the more popular cluster (training-set popularity)
    return target_rank(np.asarray(scores)[:, order], inv_order[np.asarray(y)])


bu = b.sort_values(["user_id", "pos"], kind="stable").reset_index(drop=True)
u_codes, u_start = np.unique(bu["user_id"].to_numpy(), return_index=True)
bu_item, bu_ts = bu["item_id"].to_numpy(), bu["ts"].to_numpy()


def history_matrix(rows: pd.DataFrame) -> sp.csr_matrix:
    # (n, 100) counts of the user's last MAX_HIST bookings strictly before each row
    start = u_start[np.searchsorted(u_codes, rows["user_id"].to_numpy())]
    p = rows["pos"].to_numpy()
    s, e = start + np.maximum(0, p - MAX_HIST), start + p
    ln = e - s
    idx = np.arange(ln.sum()) - np.repeat(np.cumsum(ln) - ln - s, ln)
    r = np.repeat(np.arange(len(rows)), ln)
    return sp.csr_matrix((np.ones(len(idx), np.float32), (r, bu_item[idx])), shape=(len(rows), N_CLUSTERS))


HIST = {s: history_matrix(d["warm"]) for s, d in EVAL.items()}
for s, h in HIST.items():
    n_hist = np.asarray(h.sum(1)).ravel()
    assert (n_hist == np.minimum(EVAL[s]["warm"]["pos"].to_numpy(), MAX_HIST)).all(), "history length mismatch"
    print(s, "mean history length", round(float(n_hist.mean()), 3), "(01d:", final_ref["splits"][s]["mean_history_len"], ")")
    assert round(float(n_hist.mean()), 3) == final_ref["splits"][s]["mean_history_len"]

ranks = {s: {"warm": {}, "cold": {}} for s in EVAL}
GLOB = "global popularity (warm: all training bookings; cold: first bookings)"
for s in EVAL:
    ranks[s]["warm"][GLOB] = rank_pop_ties(np.tile(pop_share, (len(Y[s]["warm"]), 1)), Y[s]["warm"])
    ranks[s]["cold"][GLOB] = rank_pop_ties(np.tile(pop_first_share, (len(Y[s]["cold"]), 1)), Y[s]["cold"])
    print(s, "warm", summarize(ranks[s]["warm"][GLOB], ks=(5,)), "cold", summarize(ranks[s]["cold"][GLOB], ks=(5,)))

valid mean history length 7.305 (01d: 7.305 )
test mean history length 7.166 (01d: 7.166 )


valid warm {'n': 213869, 'recall@5': 0.1411, 'ndcg@5': 0.0916} cold {'n': 50769, 'recall@5': 0.125, 'ndcg@5': 0.0799}


test warm {'n': 213869, 'recall@5': 0.1487, 'ndcg@5': 0.0944} cold {'n': 49462, 'recall@5': 0.1318, 'ndcg@5': 0.0814}


## 3. Baseline 1: ItemKNN (history only)

Fit on the user x cluster booking counts of the training set; `K` chosen on valid warm NDCG@10 (ties to the smaller `K`).

In [4]:
ui = sp.csr_matrix(
    (np.ones(len(train_rows), np.float32),
     (pd.factorize(train_rows["user_id"])[0], train_rows["item_id"].to_numpy())),
    shape=(train_rows["user_id"].nunique(), N_CLUSTERS),
)
ui.sum_duplicates()
print("fit matrix", ui.shape, "nnz", ui.nnz)

knn_models, knn_valid = {}, {}
for k_nb in K_GRID:
    m = CosineRecommender(K=k_nb)
    m.fit(ui, show_progress=False)
    knn_models[k_nb] = m
    r = rank_pop_ties((HIST["valid"] @ m.similarity).toarray(), Y["valid"]["warm"])
    knn_valid[k_nb] = {"monitor": ndcg_at_k(r, 10), **summarize(r, ks=KS)}
    print(k_nb, knn_valid[k_nb])
K_PICK = max(K_GRID, key=lambda k_nb: (knn_valid[k_nb]["monitor"], -k_nb))
print("picked K =", K_PICK)

KNN = f"ItemKNN (cosine, K={K_PICK}, history only; cold: first-booking popularity)"
for s in EVAL:
    ranks[s]["warm"][KNN] = rank_pop_ties((HIST[s] @ knn_models[K_PICK].similarity).toarray(), Y[s]["warm"])
    ranks[s]["cold"][KNN] = ranks[s]["cold"][GLOB]  # no history: first-booking popularity

fit matrix (713754, 100) nnz 1963673


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.016866207122802734 seconds
  warnings.warn(


5 {'monitor': 0.24851687722132895, 'n': 213869, 'recall@5': 0.3009, 'recall@10': 0.4248, 'recall@20': 0.5642, 'ndcg@5': 0.2086, 'ndcg@10': 0.2485, 'ndcg@20': 0.2837}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.02908778190612793 seconds
  warnings.warn(


10 {'monitor': 0.2512236321715708, 'n': 213869, 'recall@5': 0.3027, 'recall@10': 0.4325, 'recall@20': 0.5896, 'ndcg@5': 0.2094, 'ndcg@10': 0.2512, 'ndcg@20': 0.2909}


...

20 {'monitor': 0.2529773851688338, 'n': 213869, 'recall@5': 0.304, 'recall@10': 0.4367, 'recall@20': 0.6055, 'ndcg@5': 0.2103, 'ndcg@10': 0.253, 'ndcg@20': 0.2955}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0183563232421875 seconds
  warnings.warn(


50 {'monitor': 0.25361830535111746, 'n': 213869, 'recall@5': 0.3047, 'recall@10': 0.439, 'recall@20': 0.6112, 'ndcg@5': 0.2104, 'ndcg@10': 0.2536, 'ndcg@20': 0.297}


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0184171199798584 seconds
  warnings.warn(


100 {'monitor': 0.2534839693389824, 'n': 213869, 'recall@5': 0.3047, 'recall@10': 0.4385, 'recall@20': 0.6113, 'ndcg@5': 0.2104, 'ndcg@10': 0.2535, 'ndcg@20': 0.297}
picked K = 50


## 4. Baseline 2: multinomial logistic regression (destination + trip context, no history), early stopping

One SAGA pass per `fit` call with `warm_start=True`; after each pass valid warm NDCG@10 and valid NLL are computed. The same `EarlyStopper` as 01d decides when to stop and which pass is best; the best pass's coefficients are restored. A first booking is a normal training row here (its query context and its cluster).

In [5]:
def lr_frame(d: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame(index=d.index)
    f["stay_bucket"] = d["ctx_stay_nights"].clip(0, STAY_CLIP)
    f["lead_bucket"] = pd.cut(d["ctx_lead_days"], LEAD_BINS, labels=False)
    for c in LR_FEATURES:
        if c not in f:
            f[c] = d[c]
    # fields nulled by cleaning -> own category -1
    return f[LR_FEATURES].astype("Float64").fillna(-1).astype("int64").astype(str)


enc = OneHotEncoder(handle_unknown="ignore", dtype=np.float64)
X_tr = enc.fit_transform(lr_frame(train_rows))
y_tr = train_rows["item_id"].to_numpy()
X_ev = {s: {k: enc.transform(lr_frame(v)) for k, v in d.items()} for s, d in EVAL.items()}
print("train X", X_tr.shape, "| classes", len(np.unique(y_tr)))


def fit_lr(c_val: float):
    m = LogisticRegression(solver="saga", C=c_val, max_iter=1, tol=0.0, warm_start=True, random_state=SEED)
    stopper, rows, best_state = EarlyStopper(PATIENCE, MIN_DELTA), [], None
    for epoch in range(1, MAX_EPOCHS + 1):
        t = time.time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            m.fit(X_tr, y_tr)
        p = m.predict_proba(X_ev["valid"]["warm"])
        r = rank_pop_ties(p, Y["valid"]["warm"])
        row = {"epoch": epoch, "monitor": ndcg_at_k(r, 10), "valid_nll": round(mean_nll(np.log(p + 1e-12), Y["valid"]["warm"]), 4),
               "seconds": round(time.time() - t, 1), **summarize(r, ks=KS)}
        if stopper.update(epoch, row["monitor"]):
            best_state = (m.coef_.copy(), m.intercept_.copy())
        row["is_best"] = False
        rows.append(row)
        print(f"C={c_val} pass {epoch}: valid NLL {row['valid_nll']:.4f} NDCG@10 {row['ndcg@10']:.4f} ({row['seconds']}s) | best {stopper.best_epoch}, no gain {stopper.bad_epochs}/{PATIENCE}")
        if stopper.should_stop:
            break
    for r_ in rows:
        r_["is_best"] = r_["epoch"] == stopper.best_epoch
    m.coef_, m.intercept_ = best_state
    reason = (f"early stop after pass {rows[-1]['epoch']}" if stopper.should_stop else f"reached max_epochs = {MAX_EPOCHS}")
    if stopper.best_epoch == MAX_EPOCHS:
        print(f"WARNING: C={c_val}: best pass is the last allowed one - possibly under-trained")
    return m, rows, stopper, reason


lr_models, lr_rows, lr_stop, lr_reason = {}, {}, {}, {}
for c_val in C_GRID:
    lr_models[c_val], lr_rows[c_val], lr_stop[c_val], lr_reason[c_val] = fit_lr(c_val)
C_PICK = max(C_GRID, key=lambda c_val: (lr_stop[c_val].best_score, -c_val))
print("picked C =", C_PICK, "| best pass", lr_stop[C_PICK].best_epoch, "|", lr_reason[C_PICK])

joblib.dump({"K_PICK": K_PICK, "knn_similarity": knn_models[K_PICK].similarity, "C_PICK": C_PICK, "lr": lr_models[C_PICK], "lr_encoder": enc,
             "lr_rows": lr_rows, "knn_valid": knn_valid}, RANKS_DIR / f"baselines_es_models{SUF}.joblib")  # kept so a failed later cell never costs the training
LR = f"Logistic regression (C={C_PICK}, pass {lr_stop[C_PICK].best_epoch}, destination + context, no history)"
for s in EVAL:
    for k in ("warm", "cold"):
        ranks[s][k][LR] = rank_pop_ties(lr_models[C_PICK].predict_proba(X_ev[s][k]), Y[s][k])

train X (2424706, 34877) | classes 100


C=0.1 pass 1: valid NLL 3.2748 NDCG@10 0.3790 (19.0s) | best 1, no gain 0/2


C=0.1 pass 2: valid NLL 3.2456 NDCG@10 0.3854 (18.6s) | best 2, no gain 0/2


C=0.1 pass 3: valid NLL 3.2388 NDCG@10 0.3868 (19.6s) | best 3, no gain 0/2


C=0.1 pass 4: valid NLL 3.2370 NDCG@10 0.3873 (19.1s) | best 3, no gain 1/2


...

C=10.0 pass 5: valid NLL 3.1282 NDCG@10 0.3985 (18.7s) | best 5, no gain 0/2


C=10.0 pass 6: valid NLL 3.1249 NDCG@10 0.3992 (19.2s) | best 5, no gain 1/2


C=10.0 pass 7: valid NLL 3.1232 NDCG@10 0.3996 (18.6s) | best 7, no gain 0/2


C=10.0 pass 8: valid NLL 3.1224 NDCG@10 0.3999 (18.5s) | best 7, no gain 1/2


C=10.0 pass 9: valid NLL 3.1221 NDCG@10 0.4001 (18.5s) | best 7, no gain 2/2
picked C = 1.0 | best pass 8 | early stop after pass 10


## 5. Test and valid results (warm, cold, all events)

In [6]:
MODELS = [GLOB, KNN, LR]
COLS = ["n", "recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
results = {}
for s in EVAL:
    results[s] = {}
    for name in MODELS:
        rw, rc = ranks[s]["warm"][name], ranks[s]["cold"][name]
        results[s][name] = {"warm": summarize(rw, ks=KS), "cold (L=0)": summarize(rc, ks=KS),
                            "all events": summarize(np.concatenate([rw, rc]), ks=KS)}
for s in ("test", "valid"):
    for sl in ("warm", "cold (L=0)", "all events"):
        print(f"\n{s.upper()} | {sl} | Expedia collapsed bookings, full ranking over 100 clusters")
        print(pd.DataFrame({n: results[s][n][sl] for n in MODELS}).T[COLS].to_string())


TEST | warm | Expedia collapsed bookings, full ranking over 100 clusters
                                                                               n  recall@5  ndcg@5  recall@10  ndcg@10  recall@20  ndcg@20
global popularity (warm: all training bookings; cold: first bookings)   213869.0    0.1487  0.0944     0.2488   0.1263     0.3968   0.1633
ItemKNN (cosine, K=50, history only; cold: first-booking popularity)    213869.0    0.3036  0.2089     0.4383   0.2522     0.6097   0.2954
Logistic regression (C=1.0, pass 8, destination + context, no history)  213869.0    0.4969  0.3354     0.6897   0.3978     0.8578   0.4406

TEST | cold (L=0) | Expedia collapsed bookings, full ranking over 100 clusters
                                                                              n  recall@5  ndcg@5  recall@10  ndcg@10  recall@20  ndcg@20
global popularity (warm: all training bookings; cold: first bookings)   49462.0    0.1318  0.0814     0.2288   0.1123     0.3590   0.1449
ItemKNN (cosin

## 6. Verification (assert on failure)

In [7]:
checks = {}
rs = np.random.default_rng(SEED)
bad = [c for c in LR_FEATURES if any(f in c for f in FORBIDDEN)]  # a. no forbidden column reaches the logistic regression
checks["lr_forbidden_features"] = bad
assert not bad

eval_src = set(pd.concat([d[k] for d in EVAL.values() for k in d])["src_row"])  # b. no eval row in the training set
checks["eval_rows_in_training_data"] = int(train_rows["src_row"].isin(eval_src).sum())
assert checks["eval_rows_in_training_data"] == 0
checks["training_max_event_minus_first_valid_event"] = int(train_rows["event"].max() - EVAL["valid"]["warm"]["event"].min())
assert checks["training_max_event_minus_first_valid_event"] < 0

h_te = HIST["test"]  # c. ItemKNN: history @ similarity equals implicit's own recommend()
idx = np.sort(rs.choice(h_te.shape[0], 2000, replace=False))
dense = (h_te[idx] @ knn_models[K_PICK].similarity).toarray()
ids, sc = knn_models[K_PICK].recommend(np.arange(len(idx)), h_te[idx].tocsr(), N=N_CLUSTERS, filter_already_liked_items=False)
lib = np.zeros_like(dense)
ok = sc > -1e30  # implicit pads with -FLT_MAX and arbitrary ids when fewer than N items score: keep only real scores
row_i, col_i = np.nonzero(ok)
lib[row_i, ids[row_i, col_i]] = sc[row_i, col_i]
checks["itemknn_max_abs_diff_vs_recommend"] = float(np.abs(lib - dense).max())
assert checks["itemknn_max_abs_diff_vs_recommend"] < 1e-4


def loop_metrics(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:  # d. plain Python loop, no nbp code
    rec = nd = 0.0
    for srow, t in zip(score, y):
        top = sorted(range(N_CLUSTERS), key=lambda c: (-srow[c], -pop[c], c))[:5]
        if t in top:
            rec += 1.0
            nd += 1.0 / np.log2(top.index(t) + 2)
    return rec / len(y), nd / len(y)


y_s = Y["test"]["warm"][idx]
for name, score in ((KNN, dense), (LR, lr_models[C_PICK].predict_proba(X_ev["test"]["warm"][idx]))):
    lp_r, lp_n = loop_metrics(score, y_s)
    r = ranks["test"]["warm"][name][idx]
    fast = (float((r <= 5).mean()), float(((r <= 5) / np.log2(r + 1)).mean()))
    checks[f"loop_vs_fast [{name}]"] = {"loop": [round(lp_r, 6), round(lp_n, 6)], "fast": [round(fast[0], 6), round(fast[1], 6)]}
    assert abs(lp_r - fast[0]) < 1e-9 and abs(lp_n - fast[1]) < 1e-9

perm = rs.permutation(len(Y["test"]["warm"]))  # e. controls on test warm rows
r_shuf_lr = rank_pop_ties(lr_models[C_PICK].predict_proba(X_ev["test"]["warm"][perm]), Y["test"]["warm"])
r_shuf_knn = rank_pop_ties((h_te[perm] @ knn_models[K_PICK].similarity).toarray(), Y["test"]["warm"])
checks["controls_test_warm"] = {
    "LR real": summarize(ranks["test"]["warm"][LR], ks=(5,)), "LR shuffled query features": summarize(r_shuf_lr, ks=(5,)),
    "ItemKNN real": summarize(ranks["test"]["warm"][KNN], ks=(5,)), "ItemKNN shuffled histories": summarize(r_shuf_knn, ks=(5,)),
    "global popularity": summarize(ranks["test"]["warm"][GLOB], ks=(5,)),
}
cw = checks["controls_test_warm"]
assert cw["LR shuffled query features"]["ndcg@5"] < cw["LR real"]["ndcg@5"]
assert cw["ItemKNN shuffled histories"]["ndcg@5"] < cw["ItemKNN real"]["ndcg@5"]


def tie_spread(score: np.ndarray, y: np.ndarray) -> dict:  # f. optimistic vs pessimistic rank: how far the tie rule can move NDCG@5
    st = score[np.arange(len(y)), y][:, None]
    opt = 1 + (score > st).sum(1)
    pes = opt + (score == st).sum(1) - 1
    return {"rows_with_tie_at_target": round(float((pes > opt).mean()), 4),
            "ndcg@5_optimistic": round(float(((opt <= 5) / np.log2(opt + 1)).mean()), 4),
            "ndcg@5_pessimistic": round(float(((pes <= 5) / np.log2(pes + 1)).mean()), 4)}


checks["ties_test_warm"] = {
    KNN: tie_spread((h_te @ knn_models[K_PICK].similarity).toarray(), Y["test"]["warm"]),
    LR: tie_spread(lr_models[C_PICK].predict_proba(X_ev["test"]["warm"]), Y["test"]["warm"]),
}
print(json.dumps(checks, indent=1))
print("all verification checks passed")

{
 "lr_forbidden_features": [],
 "eval_rows_in_training_data": 0,
 "training_max_event_minus_first_valid_event": -1,
 "itemknn_max_abs_diff_vs_recommend": 0.0,
 "loop_vs_fast [ItemKNN (cosine, K=50, history only; cold: first-booking popularity)]": {
  "loop": [
   0.2935,
   0.20115
  ],
  "fast": [
   0.2935,
   0.20115
  ]
 },
 "loop_vs_fast [Logistic regression (C=1.0, pass 8, destination + context, no history)]": {
  "loop": [
   0.4905,
   0.333577
  ],
  "fast": [
   0.4905,
   0.333577
  ]
 },
 "controls_test_warm": {
  "LR real": {
   "n": 213869,
   "recall@5": 0.4969,
   "ndcg@5": 0.3354
  },
  "LR shuffled query features": {
   "n": 213869,
   "recall@5": 0.0794,
   "ndcg@5": 0.0476
  },
  "ItemKNN real": {
   "n": 213869,
   "recall@5": 0.3036,
   "ndcg@5": 0.2089
  },
  "ItemKNN shuffled histories": {
   "n": 213869,
   "recall@5": 0.0842,
   "ndcg@5": 0.0497
  },
  "global popularity": {
   "n": 213869,
   "recall@5": 0.1487,
   "ndcg@5": 0.0944
  }
 },
 "ties_test_warm":

## 7. Comparison with the plain SMLP4Rec (01e) and the final model (01d)

All rows: same training set (warm + first bookings), same test rows, full ranking over 100 clusters, test split, seed 2022 for the learned models. Numbers are read from the result JSONs; nothing is typed here. The final model is evaluated at its best epoch by valid NDCG@10.

In [8]:
plain_es = json.loads(PLAIN_ES_JSON.read_text(encoding="utf-8"))
cold_json = json.loads((FINAL_COLD_JSON).read_text(encoding="utf-8"))["results"]["test"]
bt = final_ref["samedest_hybrid"]["benchmark_test"]
ROWSETS = {"warm": "warm", "cold (L=0)": "cold", "all events": "all_events"}
cmp = {}
for name in MODELS:
    cmp[name] = {rs_: results["test"][name][rs_] for rs_ in ROWSETS}
cmp[f"SMLP4Rec plain, no query token (01e, best epoch {plain_es['best_epoch_by_valid']})"] = {rs_: plain_es["test"][rs_] for rs_ in ROWSETS}
cmp[f"SMLP4Rec + query token (final, best epoch {final_ref['best_epoch_by_valid']})"] = {
    "warm": bt["warm"]["Query token"], "cold (L=0)": cold_json["query token alone (empty history)"], "all events": bt["all_events"]["Query token"]}
cmp[f"SMLP4Rec + query token + sameDest (final, best epoch {final_ref['best_epoch_by_valid']})"] = {
    "warm": bt["warm"]["Query token + sameDest"],
    "cold (L=0)": cold_json["query token alone (empty history)"],  # no history: sameDest weight for a new destination is 0, ranking unchanged
    "all events": bt["all_events"]["Query token + sameDest"]}
for rs_ in ROWSETS:
    print(f"\nTEST | {rs_} | n = {cmp[MODELS[0]][rs_]['n']}")
    print(pd.DataFrame({n: cmp[n][rs_] for n in cmp}).T[["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]].to_string())
assert all(cmp[n][rs_]["n"] == cmp[MODELS[0]][rs_]["n"] for n in cmp for rs_ in ROWSETS), "row sets differ between models"


TEST | warm | n = 213869
                                                                        recall@5  ndcg@5  recall@10  ndcg@10  recall@20  ndcg@20
global popularity (warm: all training bookings; cold: first bookings)     0.1487  0.0944     0.2488   0.1263     0.3968   0.1633
ItemKNN (cosine, K=50, history only; cold: first-booking popularity)      0.3036  0.2089     0.4383   0.2522     0.6097   0.2954
Logistic regression (C=1.0, pass 8, destination + context, no history)    0.4969  0.3354     0.6897   0.3978     0.8578   0.4406
SMLP4Rec plain, no query token (01e, best epoch 2)                        0.3214  0.2277     0.4532   0.2701     0.6230   0.3129
SMLP4Rec + query token (final, best epoch 8)                              0.5965  0.4248     0.7723   0.4818     0.9107   0.5171
SMLP4Rec + query token + sameDest (final, best epoch 8)                   0.6104  0.4461     0.7786   0.5007     0.9125   0.5348

TEST | cold (L=0) | n = 49462
                                        

## 8. Save

In [9]:
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()
out = {
    "run": "two baselines (ItemKNN, multinomial logistic regression) on the training set of the final model (warm + first bookings), early stopping, notebook 07b",
    "dataset": "expedia (bookings, burst repeats collapsed; item = hotel_cluster)",
    "data_version": "collapsed (expedia_bookings.parquet)",
    "git_commit": commit,
    "generated_by": "notebooks/hospitality/smlp4rec/07b_baselines_same_train.ipynb",
    "libraries": {"implicit": implicit.__version__, "scikit-learn": sklearn.__version__},
    "training_set": {"rows": int(len(train_rows)), "warm_targets": int(counts["train"]), "first_bookings": int(counts["train_first"]),
                     "rule": "every booking up to the last train target; identical to 01d"},
    "split_counts": {k: int(v) for k, v in counts.items()},
    "early_stop": {"monitor": "valid warm ndcg@10", "patience": PATIENCE, "min_delta": MIN_DELTA, "max_epochs": MAX_EPOCHS},
    "models": {
        KNN: {"library": "implicit.nearest_neighbours.CosineRecommender", "input": f"last {MAX_HIST} bookings before the target",
              "cold_rows": "first-booking popularity of the training set", "K_grid_valid": knn_valid, "K_pick": K_PICK},
        LR: {"library": "sklearn LogisticRegression(solver='saga'), multinomial, one pass per fit with warm_start", "features": LR_FEATURES,
             "lead_bins": LEAD_BINS, "stay_clip": STAY_CLIP, "C_pick": C_PICK, "best_pass": lr_stop[C_PICK].best_epoch,
             "stop_reason": {str(c): lr_reason[c] for c in C_GRID}, "per_pass_valid": {str(c): lr_rows[c] for c in C_GRID}},
    },
    "selection": "valid warm NDCG@10 (K, C and pass); ties toward the smaller K / C",
    "tie_rule": "ties broken toward the cluster more popular in the training set",
    "results": results,
    "verification": checks,
}
OUT_JSON.write_text(json.dumps(out, indent=2, default=str), encoding="utf-8")
CMP_JSON.write_text(json.dumps({"split": "test", "seed": SEED, "models": cmp,
                                "note": "learned models at their best epoch by valid NDCG@10; cold sameDest = query token alone (no history)"}, indent=2, default=str), encoding="utf-8")
for s in EVAL:
    pd.concat([
        pd.DataFrame({"src_row": EVAL[s][k]["src_row"].to_numpy(), "slice": k, **{name: ranks[s][k][name].astype(np.int16) for name in MODELS}})
        for k in ("warm", "cold")
    ]).to_parquet(RANKS_DIR / f"baselines_es_ranks_{s}{SUF}.parquet", index=False)
print("saved", OUT_JSON.relative_to(ROOT), "and", CMP_JSON.relative_to(ROOT))

saved results\week4_rebuild\baselines_es_run.json and results\week4_rebuild\baselines_vs_final_es.json
